In [10]:
# install libraries if not already installed
# pip install pandas pandas_datareader fredapi matplotlib seaborn numpy python-dotenv

import pandas as pd
import pandas_datareader as pdr
from fredapi import Fred
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv


In [11]:

# Load environment variables
load_dotenv()

# Configure FRED API
fred_key = os.getenv('FRED_API_KEY')
if not fred_key:
    raise ValueError("FRED_API_KEY not found in environment variables.")
fred = Fred(api_key=fred_key)

# Create output directory
output_dir = 'data_csv'
os.makedirs(output_dir, exist_ok=True)

In [12]:
# --- Date Configuration ---

# 1. VAR Analysis Data
# Timeframe: 1987 to mid-2007 (pre-crisis)
VAR_START_DATE = '1987-01-01'
VAR_END_DATE = '2007-07-01'

# 2. Visuals & Analysis Data
# Timeframe: 2000 to present (for broader context)
VISUALS_START_DATE = '2000-01-01'
VISUALS_END_DATE = '2025-08-01'

# 3. Specific Series Overrides
HOUSING_STARTS_START = '1959-01-01' # Long-term historical context

In [13]:
# --- Ticker Definitions ---

# National Level Data
national_tickers = {
    # VAR Data Series
    'cs': 'CSUSHPINSA',       # Case-Shiller National Home Price Index
    'cs10': 'SPCS10RSA',      # Case-Shiller 10-City Home Price Index
    'ffr': 'FEDFUNDS',        # Federal Funds Rate
    'unemp': 'UNRATE',        # Unemployment Rate
    'pce_rate': 'PCETRIM12M159SFRBDAL', # PCE 12-Month Rate of Change
    
    # Visuals Data Series
    'permits': 'PERMIT',      # Building Permits
    'population': 'POPTHM',   # Population (thousands)
    'housing starts': 'HOUST',# Housing Starts
    'recessions': 'USREC',    # Recession Indicator
}

In [14]:
# Dictionary of tickers for the top 20 Metropolitan Statistical Areas (MSAs)
# HPI: S&P/Case-Shiller Home Price Index (Seasonally Adjusted)
# Permits: New Private Housing Units Authorized by Building Permits (Seasonally Adjusted)
# Population: Resident Population (Annual, interpolated to Quarterly if needed)

metropol_tickers = {
    
    'Charlotte': {'hpi': 'CRXRSA', 'permits': 'CHAR737BPPRIVSA', 'pop': 'CGRPOP'},
    'Cleveland': {'hpi': 'CEXRSA', 'permits': 'CLEV439BPPRIVSA', 'pop': 'CVLPOP'},
    'Denver': {'hpi': 'DNXRSA', 'permits': 'DENV708BPPRIVSA', 'pop': 'DNVPOP'},
    'Las Vegas': {'hpi': 'LVXRSA', 'permits': 'LASV832BPPRIVSA', 'pop': 'LSVPOP'},
    'New York': {'hpi': 'NYXRSA', 'permits': 'NEWY636BPPRIVSA', 'pop': 'NYTPOP'},
    
}

# Note: Some Case-Shiller indices start later than 1987.
# Data availability for Permits and Population varies by MSA.

In [15]:
# --- Data Fetching Helper ---
def fetch_fred_series(ticker, start, end, freq=None):
    try:
        series = fred.get_series(ticker, observation_start=start, observation_end=end)
        if freq:
            series = series.resample(freq).ffill()
            # Ensure full range coverage
            full_range = pd.date_range(start=start, end=end, freq=freq)
            series = series.reindex(full_range, method='ffill')
        return series
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None


In [16]:

print("Starting Data Download...")
var_data = {}
visuals_data = {}

# 1. Process National Data
print("Processing National Data...")
for name, ticker in national_tickers.items():
    # Determine category and date range
    if name in ['permits', 'population', 'housing starts', 'recessions']:
        # Visuals Category
        start = HOUSING_STARTS_START if name in ['housing starts', 'recessions'] else VISUALS_START_DATE
        series = fetch_fred_series(ticker, start, VISUALS_END_DATE)
        if series is not None:
            visuals_data[f"National_{name}"] = series
            print(f"  Fetched {name} (Visuals)")
    else:
        # VAR Category (cs, cs10, pce, ffr, unemp)
        series = fetch_fred_series(ticker, VAR_START_DATE, VAR_END_DATE)
        if series is not None:
            var_data[name] = series
            print(f"  Fetched {name} (VAR)")

# 2. Process Metropolitan Data
print("\nProcessing Metropolitan Data...")
for city, tickers in metropol_tickers.items():
    for var_name, ticker in tickers.items():
        col_name = city.lower() if var_name == 'hpi' else f"{city}_{var_name}"
        
        if var_name in ['permits', 'pop']:
            # Visuals Category (Permits and Population)
            # Population needs resampling to monthly
            freq = 'MS' if var_name == 'pop' else None
            series = fetch_fred_series(ticker, VISUALS_START_DATE, VISUALS_END_DATE, freq=freq)
            if series is not None:
                visuals_data[col_name] = series
                print(f"  Fetched {col_name} (Visuals)")
        else:
            # VAR Category (HPI)
            series = fetch_fred_series(ticker, VAR_START_DATE, VAR_END_DATE)
            if series is not None:
                var_data[col_name] = series
                print(f"  Fetched {col_name} (VAR)")


Starting Data Download...
Processing National Data...
  Fetched cs (VAR)
  Fetched cs10 (VAR)
  Fetched ffr (VAR)
  Fetched unemp (VAR)
  Fetched pce_rate (VAR)
  Fetched permits (Visuals)
  Fetched population (Visuals)
  Fetched housing starts (Visuals)
  Fetched recessions (Visuals)

Processing Metropolitan Data...
  Fetched charlotte (VAR)
  Fetched Charlotte_permits (Visuals)
  Fetched Charlotte_pop (Visuals)
  Fetched cleveland (VAR)
  Fetched Cleveland_permits (Visuals)
  Fetched Cleveland_pop (Visuals)
  Fetched denver (VAR)
  Fetched Denver_permits (Visuals)
  Fetched Denver_pop (Visuals)
  Fetched las vegas (VAR)
  Fetched Las Vegas_permits (Visuals)
  Fetched Las Vegas_pop (Visuals)
  Fetched new york (VAR)
  Fetched New York_permits (Visuals)
  Fetched New York_pop (Visuals)


In [17]:

# --- Save Data ---
df_var = pd.DataFrame(var_data)
df_visuals = pd.DataFrame(visuals_data)

# Adjust units if necessary
if 'National_permits' in df_visuals.columns:
    df_visuals['National_permits'] *= 1000


In [18]:

# Save to CSV
var_csv_path = os.path.join(output_dir, 'var_data.csv')
visuals_csv_path = os.path.join(output_dir, 'visual_data.csv')

df_var.to_csv(var_csv_path)
df_visuals.to_csv(visuals_csv_path)

print(f"\nSaved VAR data to {var_csv_path}")
print(f"Saved Visuals data to {visuals_csv_path}")

display(df_var.head())
display(df_visuals.head())


Saved VAR data to data_csv\var_data.csv
Saved Visuals data to data_csv\visual_data.csv


,cs,cs10,ffr,unemp,pce_rate,charlotte,cleveland,denver,las vegas,new york
1987-01-01,63.732,63.103814,6.43,6.6,3.16,63.559024,54.057393,50.348017,66.418966,74.678440
1987-02-01,64.131,63.804972,6.10,6.6,3.22,64.028844,54.229218,50.160468,67.574198,76.043687
1987-03-01,64.467,64.425674,6.13,6.6,3.41,64.105242,54.365756,50.383478,67.850132,77.237075
1987-04-01,64.972,65.063393,6.37,6.3,3.57,64.892990,54.155370,50.744628,68.186064,78.281527
1987-05-01,65.546,65.815103,6.85,6.3,3.66,65.222790,54.726192,50.763538,67.927091,79.675331


,National_permits,National_population,National_housing starts,National_recessions,Charlotte_permits,Charlotte_pop,Cleveland_permits,Cleveland_pop,Denver_permits,Denver_pop,Las Vegas_permits,Las Vegas_pop,New York_permits,New York_pop
1959-01-01,NaN,NaN,1657.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-02-01,NaN,NaN,1667.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-03-01,NaN,NaN,1620.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-04-01,NaN,NaN,1590.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-05-01,NaN,NaN,1498.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
